# Springboard-Churn-Mini-Project.ipynb

Mini Project for the USCD Springboard Machine Learning / Data Science Online Course

## Startup cells

In [0]:
# Set environment variables for sagemaker_studio imports

import os
os.environ['DataZoneProjectId'] = 'cdilii7fbtxhk7'
os.environ['DataZoneDomainId'] = 'dzd-djbrnivkolbuxz'
os.environ['DataZoneEnvironmentId'] = '3sonsi47i0102v'
os.environ['DataZoneDomainRegion'] = 'us-east-1'

# create both a function and variable for metadata access
_resource_metadata = None

def _get_resource_metadata():
    global _resource_metadata
    if _resource_metadata is None:
        _resource_metadata = {
            "AdditionalMetadata": {
                "DataZoneProjectId": "cdilii7fbtxhk7",
                "DataZoneDomainId": "dzd-djbrnivkolbuxz",
                "DataZoneEnvironmentId": "3sonsi47i0102v",
                "DataZoneDomainRegion": "us-east-1",
            }
        }
    return _resource_metadata
metadata = _get_resource_metadata()

In [0]:
"""
Logging Configuration

Purpose:
--------
This sets up the logging framework for code executed in the user namespace.
"""

from typing import Optional


def _set_logging(log_dir: str, log_file: str, log_name: Optional[str] = None):
    import os
    import logging
    from logging.handlers import RotatingFileHandler

    level = logging.INFO
    max_bytes = 5 * 1024 * 1024
    backup_count = 5

    # fallback to /tmp dir on access, helpful for local dev setup
    try:
        os.makedirs(log_dir, exist_ok=True)
    except Exception:
        log_dir = "/tmp/kernels/"

    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_file)

    logger = logging.getLogger() if not log_name else logging.getLogger(log_name)
    logger.handlers = []
    logger.setLevel(level)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    # Rotating file handler
    fh = RotatingFileHandler(filename=log_path, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging initialized for {log_name}.")


_set_logging("/var/log/computeEnvironments/kernel/", "kernel.log")
_set_logging("/var/log/studio/data-notebook-kernel-server/", "metrics.log", "metrics")

In [0]:
import logging
from sagemaker_studio import ClientConfig, sqlutils, sparkutils, dataframeutils

logger = logging.getLogger(__name__)
logger.info("Initializing sparkutils")
spark = sparkutils.init()
logger.info("Finished initializing sparkutils")

In [0]:
def _reset_os_path():
    """
    Reset the process's working directory to handle mount timing issues.
    
    This function resolves a race condition where the Python process starts
    before the filesystem mount is complete, causing the process to reference
    old mount paths and inodes. By explicitly changing to the mounted directory
    (/home/sagemaker-user), we ensure the process uses the correct, up-to-date
    mount point.
    
    The function logs stat information (device ID and inode) before and after
    the directory change to verify that the working directory is properly
    updated to reference the new mount.
    
    Note:
        This is executed at module import time to ensure the fix is applied
        as early as possible in the kernel initialization process.
    """
    try:
        import os
        import logging

        logger = logging.getLogger(__name__)
        logger.info("---------Before------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)

        os.chdir("/home/sagemaker-user")

        logger.info("---------After------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)
    except Exception as e:
        logger.exception(f"Failed to reset working directory: {e}")

_reset_os_path()

## Notebook

In light of SageMaker Unified Studio, I used the Visual ETL service to preprocess my data to serve the equivalent purpose.  It uses PySpark under the hood to autogenerate the code and preprocess the dataset.

As of June 2026, I could not find DataWrangler, but this allowed me to perform common data-processing operations (eg. remove null rows, sort by column, etc.), and export the resulting code to a ipynb notebook file to be loaded in Sagemaker Unified Studio and run.

To learn more about Visual ETL, see these docs: https://docs.aws.amazon.com/sagemaker-unified-studio/latest/userguide/gs-etl.html

In order to create a churn prediction model using Sagemaker Unified Studio, I am following the instructions in the first part of the tutorial in this provided link: https://aws.amazon.com/blogs/machine-learning/build-tune-and-deploy-an-end-to-end-churn-prediction-model-using-amazon-sagemaker-pipelines/

After preprocessing the data, I will use XGBoost to make the model and add hyperparameters for testing.

In [0]:
# %%configure -n default.spark
# {
#     "number_of_workers": 10,
#     "session_type": "etl",
#     "glue_version": "5.1",
#     "worker_type": "G.1X",
#     "idle_timeout": 15,
#     "timeout": 60,
#     "--enable-glue-datacatalog": "true",
#     "--enable-auto-scaling": "true",
#     "--project_s3_path": "s3://amazon-sagemaker-803542038076-us-east-1-cdilii7fbtxhk7/sys/",
#     "--redshift_iam_role": "arn:aws:iam::803542038076:role/service-role/AmazonSageMakerAdminIAMExecutionRole",
#     "--redshift_tempdir": "s3://amazon-sagemaker-803542038076-us-east-1-cdilii7fbtxhk7/sys/redshift/tmp/",
#     "--enable-lakeformation-fine-grained-access": "false"
# }
print("Sagemaker Unified Studio already comes preconfigured with Spark.")


Sagemaker Unified Studio already comes preconfigured with Spark.


In [0]:
import sys
from pyspark.context import SparkContext
from pyspark.sql import SparkSession
import pandas as pd
import pprint
from pyspark.sql.connect.functions import col
from sklearn.model_selection import train_test_split
import sagemaker
from sagemaker_studio import Project

In [0]:
# sc = SparkContext.getOrCreate()
# spark = SparkSession.builder.getOrCreate()
print("No need for SparkContext instance.")
print("Spark is already configured in Sagemaker Unified Studio.  Its version is: " + spark.version)


No need for SparkContext instance.


Spark is already configured in Sagemaker Unified Studio.  Its version is: 3.5.6-amzn-1


In [0]:
# Script generated for node S3DataSource
churn_df = spark.read.format("csv") \
    .option("inferschema", "true") \
    .option("multiLine", "true") \
    .option("header", "true") \
    .option("recursiveFileLookup", "true") \
    .option("sep", ",") \
    .load("s3://churn-predictor-dataset-bucket/storedata_total.csv")

print(type(churn_df))
# Don't put anything that isn't a csv file in the same s3 bucket as the csv dataset.
# or else converting to pandas won't work.
print("And the size of the churn dataset is: " + str(churn_df.toPandas().shape))


<class 'pyspark.sql.connect.dataframe.DataFrame'>


And the size of the churn dataset is: (30801, 15)


In [0]:
# Script generated for node DropNullTransform
# Drop any rows containing null entries
wo_null_rows_df = churn_df.dropna(how="any", thresh=None)
# As this shape function indicates, we have dropped no rows, so there were no null rows in the dataset.
print(wo_null_rows_df.toPandas().shape)

(30781, 15)


In [0]:
# Script generated for node DropDuplicatesTransform
wo_duplicates_df = wo_null_rows_df.dropDuplicates(["custid"])
# Nice, no duplicate customer ID's either.  It's a well-organized dataset thus far.
print(wo_duplicates_df.toPandas().shape)
print(wo_duplicates_df.toPandas().head())


(30769, 15)


   custid  retained     created  firstorder  ... refill  doorstep   favday  city
0  2253X7         1   1/10/2014   1/10/2014  ...      0         0  Tuesday   MAA
1  228RLR         1   9/13/2013   9/13/2013  ...      0         0   Friday   BOM
2  22KD32         1  10/20/2013  10/20/2013  ...      0         0  Tuesday   MAA
3  22MNP8         0   4/11/2011   4/11/2011  ...      0         0  Tuesday   DEL
4  22NS8Y         1   10/3/2013   10/3/2013  ...      1         0   Monday   BOM

[5 rows x 15 columns]


In [0]:
# Create a derived column: emails per unit order (esent divided by avgorder)
emails_per_order_df = wo_duplicates_df.withColumn(
    "emails_per_unit_order",
    col("esent") / col("avgorder")
)
print(emails_per_order_df.toPandas().head())


   custid  retained     created  ...   favday city  emails_per_unit_order
0  2253X7         1   1/10/2014  ...  Tuesday  MAA               0.258065
1  228RLR         1   9/13/2013  ...   Friday  BOM               0.949525
2  22KD32         1  10/20/2013  ...  Tuesday  MAA               0.800000
3  22MNP8         0   4/11/2011  ...  Tuesday  DEL               0.000000
4  22NS8Y         1   10/3/2013  ...   Monday  BOM               0.852130

[5 rows x 16 columns]


In [0]:
# Script generated for removing esent column
sans_esent_df = emails_per_order_df.drop("esent")
print(sans_esent_df.toPandas().columns)
print(sans_esent_df.toPandas().head())

Index(['custid', 'retained', 'created', 'firstorder', 'lastorder', 'eopenrate',
       'eclickrate', 'avgorder', 'ordfreq', 'paperless', 'refill', 'doorstep',
       'favday', 'city', 'emails_per_unit_order'],
      dtype='object')


   custid  retained     created  ...   favday city  emails_per_unit_order
0  2253X7         1   1/10/2014  ...  Tuesday  MAA               0.258065
1  228RLR         1   9/13/2013  ...   Friday  BOM               0.949525
2  22KD32         1  10/20/2013  ...  Tuesday  MAA               0.800000
3  22MNP8         0   4/11/2011  ...  Tuesday  DEL               0.000000
4  22NS8Y         1   10/3/2013  ...   Monday  BOM               0.852130

[5 rows x 15 columns]


In [0]:
# Final preprocessing of dataset based on tutorial code for preprocess_data:
# https://aws.amazon.com/blogs/machine-learning/build-tune-and-deploy-an-end-to-end-churn-prediction-model-using-amazon-sagemaker-pipelines/

# Needed for some final operations that are better fit for Pandas.
# (eg. one-hot-encoding and datetime)

# First, convert PySpark DataFrame to pandas DataFrame
sans_esent_pandas_df = sans_esent_df.toPandas()

# Create Column which gives the days between when the customer record was created and the first order
# Use errors='coerce' to handle invalid dates by converting them to NaT (Not a Time)
sans_esent_pandas_df['created'] = pd.to_datetime(sans_esent_pandas_df['created'], errors='coerce')
sans_esent_pandas_df['firstorder'] = pd.to_datetime(sans_esent_pandas_df['firstorder'], errors='coerce')
sans_esent_pandas_df['created_first_days_diff'] = (sans_esent_pandas_df['created'] - sans_esent_pandas_df['firstorder']).dt.days

# Drop rows with invalid dates (NaT values in the date difference column)
sans_esent_pandas_df = sans_esent_pandas_df.dropna(subset=['created_first_days_diff'])

# Drop unnecessary columns
sans_esent_pandas_df.drop(['custid', 'created', 'firstorder', 'lastorder'], axis=1, inplace=True)

# Apply one hot encoding on favday and city columns
preprocessed_df = pd.get_dummies(sans_esent_pandas_df, prefix=['favday', 'city'], columns=['favday', 'city'])
print(f"Preprocessed dataframe shape: {preprocessed_df.shape}")
print(preprocessed_df.head())

Preprocessed dataframe shape: (30757, 21)
   retained  eopenrate  eclickrate  ...  city_BOM  city_DEL  city_MAA
0         1  62.500000   50.000000  ...     False     False      True
1         1   0.000000    0.000000  ...      True     False     False
2         1  12.500000    3.125000  ...     False     False      True
3         0   0.000000    0.000000  ...     False      True     False
4         1  15.686275    7.843137  ...      True     False     False

[5 rows x 21 columns]


In [0]:
# Split the dataset into training, validation, and test sets.
train_and_validation, test, retained_tv, retained_test = train_test_split(
    preprocessed_df.drop('retained', axis=1, inplace=False),
    preprocessed_df['retained'],
    test_size=0.15,
    random_state=47
)

# Uncomment to ensure that train_test_split works as intended.
# print(train_and_validation.head())
# print(test.head())
# print(test.head())
# print(retained_test.head())


In [0]:
# join the features and labels of the non-test sets to train test split them again
to_split_again = train_and_validation.join(retained_tv)
# print(to_split_again.head())

# Now split this dataframe into the training and validation sets.
train, validation, retained_train, retained_validation = train_test_split(
    to_split_again.drop('retained', axis=1, inplace=False),
    to_split_again['retained'],
    test_size=0.15/0.85,
    random_state=47
)

# print(train.head())
# print(validation.head())
# print(retained_train.head())
# print(retained_validation.head())


In [0]:
# Get the project's S3 root path
root_dir = "s3://churn-predictor-dataset-bucket/training_data"
proj = Project()

# XGBoost expects the target column to be first
# Combine labels (target) with features, with target column first
retain_and_train = pd.concat([retained_train, train], axis=1)
retain_and_validation = pd.concat([retained_validation, validation], axis=1)
retain_and_test = pd.concat([retained_test, test], axis=1)

print(retain_and_train.head())
print(retain_and_validation.head())
print(retain_and_test.head())

# Define S3 paths for training, validation, and test datasets
train_s3_path = f"{root_dir}/train.csv"
validation_s3_path = f"{root_dir}/validation.csv"
test_s3_path = f"{root_dir}/test.csv"

print(f"\nSaving training data to: {train_s3_path}")
print(f"Saving validation data to: {validation_s3_path}")
print(f"Saving test data to: {test_s3_path}")

# Save to S3 (this will overwrite existing files automatically)
retain_and_train.to_csv(train_s3_path, index=False, header=False)
retain_and_validation.to_csv(validation_s3_path, index=False, header=False)
retain_and_test.to_csv(test_s3_path, index=False, header=False)

print("\nData saved successfully!")
print(f"Training set shape: {retain_and_train.shape}")
print(f"Validation set shape: {retain_and_validation.shape}")
print(f"Test set shape: {retain_and_test.shape}")

       retained  eopenrate  eclickrate  ...  city_BOM  city_DEL  city_MAA
30475         1   0.000000    0.000000  ...     False     False      True
27469         1   0.000000    0.000000  ...     False      True     False
22179         1  11.538462    0.000000  ...     False     False      True
18334         1  68.421053    0.000000  ...      True     False     False
27030         1  31.707317   14.634146  ...     False     False      True

[5 rows x 21 columns]
       retained  eopenrate  eclickrate  ...  city_BOM  city_DEL  city_MAA
29772         1   2.083333    0.000000  ...     False     False      True
3991          1  40.909091    9.090909  ...     False     False      True
23504         1   3.448276    3.448276  ...      True     False     False
30598         1  33.333333   18.750000  ...     False     False      True
8079          1  13.888889    0.000000  ...      True     False     False

[5 rows x 21 columns]
       retained  eopenrate  eclickrate  ...  city_BOM  city_DEL  c


Data saved successfully!
Training set shape: (21529, 21)
Validation set shape: (4614, 21)
Test set shape: (4614, 21)


From this point forward, we are going to follow the tutorial to generate a model using XGBoost and then to test it via hyperparameter tuning.

Since I am new to XGBoost and Sagemaker, the tried and true model of the blog seems like a fine place to start: https://aws.amazon.com/blogs/machine-learning/build-tune-and-deploy-an-end-to-end-churn-prediction-model-using-amazon-sagemaker-pipelines/

In [0]:
# Let's begin with some much-needed variables:

# Start a new Sagemaker session and execution role
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()

# Get the docker container associated with Sagemaker's XG-Boost model.
# To learn more about XG-Boost, refer to this: https://xgboost.readthedocs.io/en/stable/tutorials/model.html
container = sagemaker.image_uris.retrieve("xgboost", region, "0.90-2")

# Load training and validation csv's into Sagemaker-friendly training objects
s3_input_train = sagemaker.inputs.TrainingInput(
    s3_data=f"{root_dir}/train", content_type="csv")
s3_input_validation = sagemaker.inputs.TrainingInput(
    s3_data=f"{root_dir}/validation", content_type="csv")

# Establish some standard hyperparameters for the Sagemaker XGBoost training model.
# For a full list of acceptable hyperparameter variables and values, refer to this documentation here:
# https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost_hyperparameters.html
# For example:
# We use AUC (area under curve) as the evaluation metric: https://en.wikipedia.org/wiki/Receiver_operating_characteristic
# Binary Logistic Regression as the underlying algorithm for gradient descent: https://en.wikipedia.org/wiki/Logistic_regression
# The training runs for 100 rounds.
# Drop 30% of the trees for each step of the XGBoost training (it runs parallel trees in gradient descent).
# XGBoost uses the Tweedie disbribution and allows the variance to be set.  Learn more here: https://en.wikipedia.org/wiki/Tweedie_distribution
fixed_hyperparameters = {
    "eval_metric": "auc",
    "objective": "binary:logistic",
    "num_round": "100",
    "rate_drop": "0.3",
    "tweedie_variance_power": "1.4"
}

# Tree-specific hyperparameter ranges for XGBoost.  Learn more here: https://xgboost.readthedocs.io/en/release_1.2.0/parameter.html
# eta signifies the learning rate of each tree, so try a continuous range of values between 0 and 1
# min_child_weight is a threshold for each child in a tree required to survive in the learning model (throw away "unimportant" child nodes)
# alpha signifies L1-Regularization: https://en.wikipedia.org/wiki/Regularization_(mathematics)
# max_depth indicates how deep each tree is allowed to be (between 1 and 10 nodes deep)
hyperparameter_ranges = {
    "eta": sagemaker.tuner.ContinuousParameter(0, 1),
    "min_child_weight": sagemaker.tuner.ContinuousParameter(1, 10),
    "alpha": sagemaker.tuner.ContinuousParameter(0, 2),
    "max_depth": sagemaker.tuner.IntegerParameter(1, 10),
}

# Specify the underlying Estimator object and feed it the ec2 type, output, and hyperparameters established above.
# Learn more here: https://sagemaker.readthedocs.io/en/v2.16.0/api/training/estimators.html
# Only use 1 EC2 instance for training of type m1.m4.xlarge
# Save the training results in the output directory of the "shared" S3 bucket.
estimator = sagemaker.estimator.Estimator(
    container,
    role,
    instance_count=1,
    hyperparameters=fixed_hyperparameters,
    instance_type="ml.m4.xlarge",
    output_path=f"{root_dir}/output",
    sagemaker_session=sagemaker_session
)

# At last, we reach the actual hyperparameter tuner with the estimator object, the tree-specific hyperparameters, the EC2 instance
# and everything else specified above.  It will do 10 training jobs, 2 at a time.
# Learn more here: https://sagemaker.readthedocs.io/en/v2.39.0/api/training/tuner.html
objective_metric_name = "validation:auc"
tuner = sagemaker.tuner.HyperparameterTuner(
    estimator, objective_metric_name, hyperparameter_ranges, max_jobs=10, max_parallel_jobs=2
)


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


In [0]:
# And now, per the blog:
# https://aws.amazon.com/blogs/machine-learning/build-tune-and-deploy-an-end-to-end-churn-prediction-model-using-amazon-sagemaker-pipelines/
# we fit our newly created hyperparameter tuner to the training and validation data we created earlier.
tuner.fit({"train": s3_input_train, "validation": s3_input_validation}, include_cls_metadata=False)

# After 10 jobs run 2 at a time, this is the object holding all the completed tuning jobs.
tuning_job_result = boto3.client("sagemaker").describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuner.latest_tuning_job.job_name
)

job_count = tuning_job_result["TrainingJobStatusCounters"]["Completed"]
print("%d training jobs have completed" % job_count)

# Use pprint to pull out and display the hyperparameters corresponding to the best tuning job.
if tuning_job_result.get("BestTrainingJob", None):
    print("Best Model found so far:")
    pprint.pprint(tuning_job_result["BestTrainingJob"])
else:
    print("No training jobs have reported results yet.")


.

.

.

.

.

...

.

.

.

!


10 training jobs have completed
Best Model found so far:
{'CreationTime': datetime.datetime(2026, 6, 29, 2, 51, 9, tzinfo=tzlocal()),
 'FinalHyperParameterTuningJobObjectiveMetric': {'MetricName': 'validation:auc',
                                                 'Value': 0.9783239960670471},
 'ObjectiveStatus': 'Succeeded',
 'TrainingEndTime': datetime.datetime(2026, 6, 29, 2, 51, 52, tzinfo=tzlocal()),
 'TrainingJobArn': 'arn:aws:sagemaker:us-east-1:803542038076:training-job/sagemaker-xgboost-260629-0246-007-e002d5b5',
 'TrainingJobName': 'sagemaker-xgboost-260629-0246-007-e002d5b5',
 'TrainingJobStatus': 'Completed',
 'TrainingStartTime': datetime.datetime(2026, 6, 29, 2, 51, 13, tzinfo=tzlocal()),
 'TunedHyperParameters': {'alpha': '0.5556642099837628',
                          'eta': '0.33802568212818274',
                          'max_depth': '3',
                          'min_child_weight': '2.6841410737577176'}}


Using the AI tool in Sagemaker Unified Studio, it's time to measure the quality of the best XGBoost model from above against the test data.

In [0]:
import numpy as np
from sklearn.metrics import roc_auc_score
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

# Get the best training job name from the tuner
best_training_job_name = tuner.best_training_job()
print(f"Best training job: {best_training_job_name}")

# Attach to the best model from the tuning job
best_estimator = sagemaker.estimator.Estimator.attach(best_training_job_name)

# Deploy the best model to an endpoint for prediction
predictor = best_estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.t2.medium",
    serializer=CSVSerializer(),
    deserializer=CSVDeserializer()
)

# Convert boolean columns to integers (True->1, False->0)
# XGBoost expects numeric values, not boolean strings
test_numeric = test.astype(float)

# Convert to numpy array
test_array = test_numeric.values

# Make predictions on the test set
predictions = predictor.predict(test_array)

# CSVDeserializer returns [[pred1, pred2, pred3, ...]]
# We need to flatten this to get the individual predictions
if len(predictions) == 1 and isinstance(predictions[0], list):
    # Predictions are in format [[pred1, pred2, ...]]
    predicted_probabilities = np.array([float(p) for p in predictions[0]])
else:
    # Predictions are in format [[pred1], [pred2], ...]
    predicted_probabilities = np.array([float(pred[0]) for pred in predictions])

# Calculate evaluation metrics
auc = roc_auc_score(retained_test, predicted_probabilities)

print(f"AUC Score: {auc}")

# Clean up - delete the endpoint after evaluation to avoid charges
predictor.delete_endpoint()
print("\nEndpoint deleted successfully.")


Best training job: sagemaker-xgboost-260629-0246-007-e002d5b5


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix



2026-06-29 02:52:04 Starting - Found matching resource for reuse
2026-06-29 02:52:04 Downloading - Downloading the training image
2026-06-29 02:52:04 Training - Training image download completed. Training in progress.
2026-06-29 02:52:04 Uploading - Uploading generated training model
2026-06-29 02:52:04 Completed - Resource reused by training job: sagemaker-xgboost-260629-0246-009-95b9b937

-

...

-

-

!

AUC Score: 0.9771941965554483



Endpoint deleted successfully.


The print in the above cell is a bit small, but the AUC of the best model from XGBoost is approximately 0.9772.

## Shutdown cells

In [0]:
"""
Stop spark session and associated Athena Spark session
"""

from IPython import get_ipython as _get_ipython
_get_ipython().user_ns["spark"].stop()